In [1]:
#  Install the custom Jane Street package (provided by competition organizers)
import os


# Kaggle's evaluation module for inference (do not modify)
from kaggle_evaluation import jane_street_inference_server


# Import essential libraries for data & ML
import numpy as np
import pandas as pd
import polars as pl       # optional, faster dataframe library
import torch              # for deep learning models
from tqdm import tqdm
import pickle

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

import statsmodels.api as sm
from sklearn.linear_model import ElasticNetCV

In [2]:
features = pd.read_csv('/kaggle/input/jane-street-real-time-market-data-forecasting/features.csv')
responders = pd.read_csv('/kaggle/input/jane-street-real-time-market-data-forecasting/responders.csv')
sample_submission = pd.read_csv('/kaggle/input/jane-street-real-time-market-data-forecasting/sample_submission.csv')

test_parquet = pl.read_parquet('/kaggle/input/jane-street-real-time-market-data-forecasting/test.parquet/date_id=0/part-0.parquet')

lags_parquet = pl.read_parquet('/kaggle/input/jane-street-real-time-market-data-forecasting/lags.parquet/date_id=0/part-0.parquet')

# 전체 train_parquet, 이 중 partition_id = 6 까지만 사용할 것
train_parquet = []
for i in tqdm(range(10), desc='train_parquet') :
    file = f'/kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/partition_id={i}/part-0.parquet'
    train_parquet.append(pl.read_parquet(file))

train_parquet: 100%|██████████| 10/10 [01:48<00:00, 10.87s/it]


In [3]:
# 합치기
parquet = train_parquet[:7]
parquets = parquet[0]
for i in parquet[1:] :
    parquets = pl.concat([parquets, i])
parquets

# symbol_id별로 나누기
symbol_ids = parquets['symbol_id'].unique()

parquets_by_symbol_id = []

for i in tqdm(symbol_ids) :
    df = parquets.filter(pl.col('symbol_id') == i)
    parquets_by_symbol_id.append(df)


100%|██████████| 39/39 [00:20<00:00,  1.88it/s]


In [4]:
# dump

#parquets_by_symbol_id
for i in tqdm(range(len(parquets_by_symbol_id))) :
    with open(f'parquets_by_symbol_id_{i}', 'wb') as f :
        pickle.dump(parquets_by_symbol_id[i], f)

100%|██████████| 39/39 [01:13<00:00,  1.88s/it]


1. 

In [5]:
# ElasticNet 기반 coefficient의 절대값 반환 함수
# X: target_features_non_null.to_numpy()
# y: target.select(pl.col('responder_6')).to_numpy().ravel()
def EN(X, y) :
    model= ElasticNetCV(alphas=np.logspace(-4, 4, 50), l1_ratio=[.1, .5, .9], cv=5)
    model.fit(X, y)
    return pl.DataFrame(dict(zip(features_list, abs(model.coef_))))

In [6]:
def coef(symbol_number) :
    # coefficient 저장용 데이터프레임 생성
    features_list = parquets_by_symbol_id[0].columns[4: -9]
    
    coefs_by_symbol = []
    
    dummy = {col:[0.0] for col in features_list}
    coefs = pl.DataFrame(dummy).clear()
    
    print(f'symbol {symbol_number}의 전체 일별 coefficient 도출')
    symbol = parquets_by_symbol_id[symbol_number]
    days = symbol['date_id'].unique().to_list()
    
    for day in tqdm(days, desc='일별 도출 중..') :
        target = symbol.filter(pl.col('date_id')==day)
        features_list = target.columns[4: -9]
        
        target_features = target.select(pl.col(features_list))
    
        target_features_non_null = target_features.select([
            pl.col(col).fill_null(pl.col(col).mean()) if target_features[col].null_count() < target_features.height
            else pl.col(col).fill_null(0)
            for col in target_features.columns
        ])
    
        target_features_non_null
        
        X = target_features_non_null.to_numpy()
        y = target.select(pl.col('responder_6')).to_numpy().ravel()
    
        coefs = coefs.vstack(EN(X, y))
        
    coefs_by_features = {}
    for col in coefs.columns :
        coefs_by_features[col] = coefs[col].sum()

    coefs_by_features_index = coefs.columns
    coefs_by_features_col = 'coef'
    coefs_by_features_row = []
    
    for col in coefs.columns :
        coefs_by_features_row.append(coefs[col].sum())

    coefs_by_features = pl.DataFrame({'index':coefs_by_features_index, 'coef': coefs_by_features_row})
    return(coefs_by_features)

In [7]:
# symbol별 coefficient 저장용 데이터프레임 생성
features_list = parquets_by_symbol_id[0].columns[4: -9]
coefs_by_symbol = pl.DataFrame({"index": features_list})

#symbol별 coefficient 도출
for i in list(symbol_ids[0:3]) :
    print(f'symbol_ids: {i}')
    coefs_by_symbol = coefs_by_symbol.with_columns(pl.Series(name=f'coef_symbol{i}', values=coef(i)['coef']))

coefs_by_symbol

symbol_ids: 0
symbol 0의 전체 일별 coefficient 도출


일별 도출 중..:  23%|██▎       | 250/1100 [01:23<06:20,  2.24it/s]/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.02654509826034257, tolerance: 0.01590443717347728
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.0337848849597151, tolerance: 0.01590443717347728
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.04032898478324398, tolerance: 0.01590443717347728
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.11/dist-

symbol_ids: 1
symbol 1의 전체 일별 coefficient 도출


일별 도출 중..:  79%|███████▉  | 941/1188 [05:20<01:25,  2.88it/s]/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.09153663281711033, tolerance: 0.0762454434483526
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.0859035041144125, tolerance: 0.07933794499666104
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:617: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.09135799433431657, tolerance: 0.07933794499666104
  model = cd_fast.enet_coordinate_descent_gram(
일별 도출 중..: 100%|██████████| 1188

symbol_ids: 2
symbol 2의 전체 일별 coefficient 도출


일별 도출 중..: 100%|██████████| 1101/1101 [06:56<00:00,  2.64it/s]


index,coef_symbol0,coef_symbol1,coef_symbol2
str,f64,f64,f64
"""feature_00""",1.268273,1.472388,1.572294
"""feature_01""",18.418755,17.163563,22.787215
"""feature_02""",2.359118,1.610248,1.588891
"""feature_03""",2.010897,1.151652,1.784199
"""feature_04""",14.57345,13.98725,16.347199
…,…,…,…
"""feature_74""",1.680499,2.656263,9.426735
"""feature_75""",1.933474,1.403098,3.996489
"""feature_76""",2.223453,1.055784,3.906453


In [8]:
fig = px.bar(x=coefs_by_symbol['index'], y=coefs_by_symbol['coef_symbol0'])
fig.update_layout(title_text="coef_symbol_id_0", title_x=0.5)
fig.show()

In [9]:
px.bar(x=coefs_by_symbol['index'], y=coefs_by_symbol['coef_symbol1'])
fig.update_layout(title_text="coef_symbol_id_1", title_x=0.5)
fig.show()

In [10]:
px.bar(x=coefs_by_symbol['index'], y=coefs_by_symbol['coef_symbol2'])
fig.update_layout(title_text="coef_symbol_id_2", title_x=0.5)
fig.show()

In [11]:
print(coefs_by_symbol.top_k(30, by='coef_symbol1')["index"].to_list())

['feature_05', 'feature_07', 'feature_58', 'feature_60', 'feature_57', 'feature_01', 'feature_37', 'feature_47', 'feature_33', 'feature_67', 'feature_56', 'feature_04', 'feature_08', 'feature_38', 'feature_72', 'feature_18', 'feature_46', 'feature_69', 'feature_65', 'feature_50', 'feature_70', 'feature_12', 'feature_49', 'feature_39', 'feature_36', 'feature_45', 'feature_06', 'feature_53', 'feature_66', 'feature_42']


In [12]:
def top30fig(symbol) :
    #symbol = 0
    coefs_by_symbol_target = coefs_by_symbol['index', f'coef_symbol{symbol}']
    top30 = coefs_by_symbol_target.top_k(30, by=f'coef_symbol{symbol}')['index']
    
    coefs_by_symbol_target
    coefs_by_symbol_target = coefs_by_symbol_target.with_columns(
        pl.when(pl.col('index').is_in(top30)).then(pl.lit('#1f77b4'))
        .otherwise(pl.lit('#7f7f7f'))#1f77b4
        .alias('color'))
    
    fig = go.Figure()
    colors = coefs_by_symbol_target['color'].to_list()
    
    fig.add_trace(go.Bar(x=coefs_by_symbol_target['index'],y=coefs_by_symbol_target[f'coef_symbol{symbol}'], marker_color=colors))
    fig.update_layout(title_text=f'coef_symbol_id_{symbol}, top30', title_x=0.5)
    fig.show()
    
    print(top30.to_list())

In [13]:
top30fig(0)

['feature_07', 'feature_05', 'feature_60', 'feature_58', 'feature_01', 'feature_57', 'feature_47', 'feature_33', 'feature_04', 'feature_06', 'feature_37', 'feature_56', 'feature_53', 'feature_08', 'feature_46', 'feature_36', 'feature_70', 'feature_72', 'feature_67', 'feature_49', 'feature_50', 'feature_39', 'feature_42', 'feature_69', 'feature_45', 'feature_38', 'feature_65', 'feature_17', 'feature_12', 'feature_18']


In [14]:
top30fig(1)

['feature_05', 'feature_07', 'feature_58', 'feature_60', 'feature_57', 'feature_01', 'feature_37', 'feature_47', 'feature_33', 'feature_67', 'feature_56', 'feature_04', 'feature_08', 'feature_38', 'feature_72', 'feature_18', 'feature_46', 'feature_69', 'feature_65', 'feature_50', 'feature_70', 'feature_12', 'feature_49', 'feature_39', 'feature_36', 'feature_45', 'feature_06', 'feature_53', 'feature_66', 'feature_42']


In [15]:
top30fig(2)

['feature_05', 'feature_58', 'feature_01', 'feature_37', 'feature_56', 'feature_47', 'feature_60', 'feature_38', 'feature_07', 'feature_57', 'feature_33', 'feature_04', 'feature_70', 'feature_45', 'feature_49', 'feature_53', 'feature_50', 'feature_08', 'feature_46', 'feature_42', 'feature_73', 'feature_78', 'feature_65', 'feature_18', 'feature_67', 'feature_77', 'feature_12', 'feature_66', 'feature_72', 'feature_39']
